# Converting documents to KPUB

Ricecooker converts article-style documents to **KPUB** (a self-contained HTML
article that Kolibri can render) automatically. Just point a `ContentNode` at your
source file with the `uri` parameter and the pipeline handles the rest — no manual
pre-conversion and no external conversion service.

The following source formats are supported:

- Word documents: `.docx`
- OpenDocument text: `.odt`
- Rich Text Format: `.rtf`
- Markdown: `.md`, `.markdown`

During conversion the pipeline extracts embedded images into the KPUB, renders math
as presentation MathML, and strips the output down to semantic HTML so it renders
cleanly in Kolibri.

## System dependency: pandoc

Document conversion shells out to [`pandoc`](https://pandoc.org/), which must be
installed on the machine running your chef (alongside `ffmpeg` and `poppler`). See
[`docs/installation.md`](../installation.md) for per-platform instructions. If
`pandoc` is missing, the pipeline raises a clear error telling you to install it.

## The workflow

Reference the source document from a `ContentNode` and it becomes a validated KPUB
in your channel:

In [ ]:
from ricecooker.classes.nodes import ContentNode
from ricecooker.classes.licenses import get_license
from le_utils.constants import licenses

# The source document lives next to your chef script (or at any local/remote URI).
# The pipeline converts it to a KPUB automatically during processing.
article = ContentNode(
    source_id="article-1",
    title="My article",
    uri="article.docx",  # .odt, .rtf, .md, and .markdown work the same way
    license=get_license(licenses.PUBLIC_DOMAIN),
)

That's the whole workflow. When the chef runs, ricecooker downloads the source
document (if remote), converts it to KPUB with pandoc, sanitizes and validates the
result, and uploads it as part of your channel.

## Out of scope

The automatic document-to-KPUB path is for **article-style** content only. The
following are intentionally not converted:

- Presentations (`.pptx`, `.ppt`) and spreadsheets (`.xlsx`, `.xls`) — not article
  content.
- Legacy binary `.doc` — pandoc cannot read the format; re-save it as `.docx` first.

For presentations and spreadsheets, convert to PDF out-of-band and add the PDF with
`ContentNode(uri="slides.pdf")`.